# Brownian Diffusion
We here show how to set up an Analysis object and use it to first fit an artificial vanadium measurement to obtain the resolution. Next, we use the fitted resolution to fit an artificial measurement of a model with diffusion and some elastic scattering. 

We extract and plot the relevant parameters and fit them to a diffusion model. Finally, we show how to fit all the data simultaneously to the diffusion model.

In [2]:
# Imports
import pooch

import easydynamics as edyn
import easydynamics.sample_model as sm

# Make the plots interactive
%matplotlib widget

We first create an `Experiment` object to contain the data. The data must either be a hdf5 file or a scipp.DataArray; in both cases it must have coordinates `Q` and `energy`. We here use Pooch to download an example vanadium data set.

The data can be rebinned if needed, but we will show how to do that in a different tutorial.

In [3]:
# Load the vanadium data
vanadium_experiment = edyn.Experiment('Vanadium')

file_path = pooch.retrieve(
    url='https://github.com/easyscience/dynamics-lib/raw/refs/heads/master/docs/docs/tutorials/data/vanadium_data_example.h5',
    known_hash='16cc1b327c303feeb88fb9dda5390dc4880b62396b1793f98c6fef0b27c7b873',
)


vanadium_experiment.load_hdf5(filename=file_path)

We can visualize the data in multiple ways, relying on plopp: https://scipp.github.io/plopp/

We here show two ways to look at the data: as a 2d colormap with intensity as function of `Q` and `energy`, and as a slicer with intensity as function of `energy` for various `Q`.

If you want $Q$ on the x axis, then set `transpose_axes=True`

In [4]:
vanadium_experiment.plot_data(slicer=False, transpose_axes=False)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [5]:
vanadium_experiment.plot_data(slicer=True)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

We now want to fit the vanadium data to determine our resolution. The scattering from vanadium is almost exclusively incoherent elastic, so we model it as a delta function. We do this by creating a `SampleModel` and adding a `DeltaFunction` component to it. The component acts as a template and gets copied to every `Q` when we attach the `SampleModel` to our `Analysis` object. Let's create the `SampleModel`.

We do not give the `DeltaFunction` a `center` value. In this case, the center will be fixed at 0 energy transfer. We set the start value of the area to 1.

In [6]:
delta_function = sm.DeltaFunction(name='DeltaFunction', area=1)
sample_model = sm.SampleModel(components=delta_function)

We now want to define our resolution function. We will here model it as a Gaussian. We create a `ComponentCollection` and append the `Gaussian` to it. We can add as many components to our resolution as we like; sometimes you need several Gaussians and other functions to accurately describe the resolution.

We fix the area of the resolution to have value 1. If we did not do this, we would fit both the area of the delta function and of the resolution Gaussian, and the fit would never converge.

We finally insert the components in a `ResolutionModel`

In [7]:
resolution_components = sm.ComponentCollection()
res_gauss = sm.Gaussian(width=0.1, area=1, name='Res. Gauss')
res_gauss.area.fixed = True
resolution_components.append_component(res_gauss)
resolution_model = sm.ResolutionModel(components=resolution_components)

The background intensity was not 0, so we also create a background model. We use a `Polynomial` with a single coefficient, i.e. a flat background. We here show how to create the `BackgroundModel` and add the background in a single line. We could of course also add it like we did for the `SampleModel` or first create a `ComponentCollection` like we did for the `ResolutionModel`

In [8]:
background_model = sm.BackgroundModel(components=sm.Polynomial(coefficients=[0.001]))

We combine the resolution abd background model into an `InstrumentModel`. This model also contains a fittable energy offset to account for instrument misalignment. All components are centered at this energy offset.

In [9]:
instrument_model = sm.InstrumentModel(
    resolution_model=resolution_model,
    background_model=background_model,
)

We are now ready to collect everything in an analysis object. We give it a display name, the experiment, the sample model and the instrument model. It will then automatically generate a model for each `Q` using the templates given in the `SampleModel`, `ResolutionModel` and `BackgroundModel`.

In [10]:
vanadium_analysis = edyn.Analysis(
    display_name='Vanadium Full Analysis',
    experiment=vanadium_experiment,
    sample_model=sample_model,
    instrument_model=instrument_model,
)

Let us first fit a single Q index and plot the data and model to see how it looks. For this, we use the `independent` fit method and choose an arbitrary Q index

In [11]:
fit_result_independent_single_Q = vanadium_analysis.fit(fit_method='independent', Q_index=5)
vanadium_analysis.plot_data_and_model(Q_index=5)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

The fit looks good, so let us fit all Q indices independently and plot the results.

In [12]:
fit_result_independent_all_Q = vanadium_analysis.fit(fit_method='independent')
vanadium_analysis.plot_data_and_model()

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [14]:
datagroup = vanadium_analysis.data_and_model_to_datagroup()
datagroup

DataGroup(sizes={'Q': 16, 'energy': 201}, keys=[
    Data: DataArray({'Q': 16, 'energy': 201}),
    Model: DataArray({'Q': 16, 'energy': 201}),
    DeltaFunction: DataArray({'Q': 16, 'energy': 201}),
    Polynomial: DataArray({'Q': 16, 'energy': 201}),
])